# End-to-end `3pwr` lifecycle notebook

A **fixed**, top-to-bottom run of the whole `3pwr` lifecycle against the sample
project in this folder, driven by [papermill](https://papermill.readthedocs.io).
It is the executable configuration for real-world testing of the CLI on this
adapter — edit the procedure *here*, nowhere else.

Each run provisions a throwaway sandbox (a git copy of `project/` with its
dependencies installed and a sandbox-scoped signer), proves the baseline gates
are green, then drives `3pwr run` through both human gates via the CLI's own
`--resume` path. The two modes:

- **full run** — `./e2e/run.sh <lang>` — dispatches the configured headless
  agent; needs its CLI + credentials.
- **check / dry run** — `./e2e/run.sh <lang> --check` — the sim runner, no agent
  and no credentials; a deterministic regression test of the kit itself.

The `3pwr` commands are shown as a plain CLI session so an agent can mimic the
procedure by hand. Committed with **cleared outputs**.


In [ ]:
# --- Parameters (papermill injects overrides just below this cell) ---
LANGUAGE = 'go'
INTENT = 'add a fixed-window rate-limiting strategy'
INTEGRATION = ""          # empty -> the coder backend from e2e/config/roles.yaml
MODE = "auto"             # auto: stop only at the two human gates
TIER = "Standard"
APPROVER = "e2e-harness"  # recorded approver for the spec + sign-off gates
KEEP_SANDBOX = False      # keep the sandbox after the run for inspection
DRY_RUN = False           # True -> sim runner, no live agent dispatch


In [ ]:
# Cell 2 — normalize parameters, locate the harness, toolchain preflight.
import json, os, shlex, subprocess, sys
from pathlib import Path


def _as_bool(value):
    """papermill's -p only coerces 'True'/'False'; accept the shell's 'true'/'false' too."""
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes", "on"}


DRY_RUN = _as_bool(DRY_RUN)
KEEP_SANDBOX = _as_bool(KEEP_SANDBOX)

# Locate the e2e/ root so we can import the shared bootstrap (no engine imports).
_env_root = os.environ.get("E2E_ROOT")
if _env_root:
    E2E_ROOT = Path(_env_root).resolve()
else:
    _here = Path.cwd().resolve()
    E2E_ROOT = None
    for _cand in [_here, *_here.parents]:
        if (_cand / "harness" / "bootstrap.py").is_file():
            E2E_ROOT = _cand
            break
        if (_cand / "e2e" / "harness" / "bootstrap.py").is_file():
            E2E_ROOT = _cand / "e2e"
            break
    if E2E_ROOT is None:
        raise RuntimeError("could not locate the e2e/ root; set E2E_ROOT")
sys.path.insert(0, str(E2E_ROOT / "harness"))
import bootstrap  # noqa: E402

print(f"language   : {LANGUAGE}")
print(f"e2e root   : {E2E_ROOT}")
print(f"dry run    : {DRY_RUN}")
print(f"intent     : {INTENT}")


def _probe(tool, args, install_hint):
    """Fail fast with the adapter's own install hint when a required tool is absent."""
    try:
        subprocess.run([tool, *args], capture_output=True, check=False)
    except FileNotFoundError:
        raise SystemExit(f"missing '{tool}'. {install_hint}") from None
    print(f"  ok  {tool}")


# 3pwr drives every stage; git is the substrate.
_probe("3pwr", ["--version"], "install the 3pwr CLI: uv tool install ./engine")
_probe("git", ["--version"], "install git")

# The adapter's toolchain (matches scaffold/adapters/<lang>/adapter.yaml).
if LANGUAGE == "typescript":
    _probe("node", ["--version"], "install Node.js 20+ — https://nodejs.org")
    _probe("npm", ["--version"], "install npm (bundled with Node.js)")
elif LANGUAGE == "python":
    _probe("uv", ["--version"], "install uv — https://docs.astral.sh/uv/")
elif LANGUAGE == "go":
    _probe("go", ["version"], "install the Go toolchain — https://go.dev/dl/")
    _probe("gcov2lcov", ["-h"], "go install github.com/jandelgado/gcov2lcov@latest")

print("preflight ok")


In [ ]:
# Cell 3 — provision a throwaway sandbox: copy project/ -> git init -> install
# deps -> 3pwr keygen + init -> seed the shared headless config. Never touches
# this repo; the sandbox lives under a temp dir.
SANDBOX = bootstrap.provision(LANGUAGE, e2e_root=E2E_ROOT)
SANDBOX_DIR = Path(SANDBOX.sandbox_dir)
ARTIFACT_DIR = Path(SANDBOX.artifact_dir)
print(f"sandbox   : {SANDBOX_DIR}")
print(f"artifacts : {ARTIFACT_DIR}")
print(f"signer    : {SANDBOX.key_file}")


def cli(command, *, expect=None, check=True):
    """Run a `3pwr`/shell command in the sandbox as a visible CLI session.

    Prints `$ <command>`, streams output, and returns the exit status. When
    `expect` is given, asserts the exit status matches (the documented run
    contract: 3 = paused at a human gate, 0 = done).
    """
    print(f"$ {command}")
    proc = subprocess.run(command, cwd=str(SANDBOX_DIR), shell=True)
    rc = proc.returncode
    print(f"[exit {rc}]")
    if expect is not None and rc != expect:
        raise AssertionError(f"expected exit {expect}, got {rc}: {command}")
    if check and expect is None and rc != 0:
        raise AssertionError(f"command failed ({rc}): {command}")
    return rc


In [ ]:
# Cell 4 — trust setup. bootstrap already ran keygen + init and seeded the
# shared headless config; assert the substrate is in place before any run.
assert (SANDBOX_DIR / ".3powers").is_dir(), "3pwr init did not lay down .3powers/"
assert Path(SANDBOX.key_file).is_file(), "sandbox signing key missing"
assert os.environ.get("THREEPOWERS_SIGNING_KEY_FILE") == SANDBOX.key_file, \
    "signing key not exported for later 3pwr commands"
assert (SANDBOX_DIR / ".3powers" / "config" / "roles.yaml").is_file(), \
    "shared headless config not seeded"
print("trust setup ok — signer, .3powers/, and shared roles.yaml in place")


In [ ]:
# Cell 5 — the standing kit invariant: the baseline gate suite is GREEN before
# any lifecycle run. A fresh sandbox has no spec yet, so this is a brownfield
# report-only run; the two spec-bound gates skip, every language gate must pass.
gate = subprocess.run(
    "3pwr gate run --report-only --tier Standard --json",
    cwd=str(SANDBOX_DIR), shell=True, capture_output=True, text=True,
)
print("$ 3pwr gate run --report-only --tier Standard --json")
verdict = json.loads(gate.stdout)["verdict"]
for g in verdict["gates"]:
    print(f"  {g['gate']:18} {g['status']:6} {g.get('tool', '')}")
reds = [g["gate"] for g in verdict["gates"] if g["status"] == "fail"]
assert verdict["result"] == "pass" and not reds, f"baseline gates not green: {reds}"
print("baseline gates green")


In [ ]:
# Cell 6 — start the lifecycle. In auto mode the run stops only at the two human
# gates. --dry-run forces the offline sim runner (no agent). The documented
# contract: a paused human gate exits 3.
DRY = " --dry-run" if DRY_RUN else ""
INTEG = f" --integration {shlex.quote(INTEGRATION)}" if INTEGRATION and not DRY_RUN else ""
cli(
    f"3pwr run {shlex.quote(INTENT)} --mode {MODE} --tier {TIER} --no-input{INTEG}{DRY}",
    expect=3,  # paused at the spec-approval human gate
)
print("paused at the spec-approval gate, as expected")


In [ ]:
# Cell 7 — HUMAN GATE 1 (spec approval). Read the generated spec, then approve
# via the CLI's own resume path (the ledger records the approver). A full run
# writes the spec to specs-src/<NNN>-<slug>/; the dry-run sim runner writes none.
specs = sorted((SANDBOX_DIR / "specs-src").glob("*/spec.md")) if (SANDBOX_DIR / "specs-src").is_dir() else []
if specs:
    spec_md = specs[-1]
    print(f"--- generated spec: {spec_md.relative_to(SANDBOX_DIR)} ---")
    print(spec_md.read_text()[:4000])
else:
    print("(dry run: the sim runner authors no on-disk spec to render)")

cli(
    f"3pwr run --resume --spec-id RUN --approver {shlex.quote(APPROVER)}{DRY}",
    expect=3,  # advances through Plan/Build/Verify, pauses at the sign-off gate
)
print("spec approved; paused at the sign-off gate")


In [ ]:
# Cell 8 — HUMAN GATE 2 (sign-off). Review the verdict, then sign off via the
# same resume path. This completes the lifecycle (exit 0).
verdict_path = SANDBOX_DIR / ".3powers" / "verdicts" / "latest.json"
if verdict_path.is_file():
    v = json.loads(verdict_path.read_text()).get("verdict", {})
    print(f"verify verdict: result={v.get('result')} tier={v.get('tier')}")

cli(
    f"3pwr run --resume --spec-id RUN --approver {shlex.quote(APPROVER)}{DRY}",
    expect=0,  # lifecycle complete
)
print("signed off; lifecycle complete")


In [ ]:
# Cell 9 — post-run assertions. INVARIANTS ONLY — never assert exact
# agent-authored content (it differs every run).
cli("3pwr verify", expect=0)                       # ledger chain + signatures intact
status = subprocess.run(
    "3pwr status --json", cwd=str(SANDBOX_DIR), shell=True,
    capture_output=True, text=True,
)
print("$ 3pwr status")
rows = json.loads(status.stdout)
assert any(r.get("signed_off") or r.get("stage") in ("Ship", "Observe") for r in rows), \
    "status does not show a completed/signed-off run"
print("verify green; status shows the completed run")

if not DRY_RUN:
    # The feature folder holds the stage artifacts an agent-driven run produced.
    feature_dirs = [p for p in (SANDBOX_DIR / "specs-src").glob("*/") if (p / "spec.md").is_file()]
    assert feature_dirs, "expected specs-src/<NNN>-<slug>/ with stage artifacts"
    print(f"feature folder: {feature_dirs[-1].relative_to(SANDBOX_DIR)}")

    # One cheap behavioral smoke — the fixed-window strategy appears in the package source.
    grep = subprocess.run(
        ["grep", "-rli", "--exclude-dir=.git", "--exclude-dir=.3powers",
         "--exclude-dir=node_modules", "--exclude-dir=.venv", "--exclude-dir=coverage",
         "fixed", str(SANDBOX_DIR)],
        capture_output=True, text=True,
    )
    assert grep.returncode == 0, "behavioral smoke failed: the fixed-window strategy appears in the package source"
    print("behavioral smoke ok — the fixed-window strategy appears in the package source")
else:
    print("(dry run: skipping artifact + behavioral-smoke assertions)")


In [ ]:
# Cell 10 — teardown. Print the path first, then remove the sandbox unless the
# run asked to keep it. The executed copy of this notebook + logs (written by
# run.sh) already live under the artifact dir, never back in the repo.
import shutil
print(f"sandbox    : {SANDBOX_DIR}")
print(f"artifacts  : {ARTIFACT_DIR}")
if KEEP_SANDBOX:
    print("KEEP_SANDBOX set — leaving the sandbox in place for inspection")
else:
    shutil.rmtree(SANDBOX_DIR.parent, ignore_errors=True)
    print("sandbox removed")
print("done")
